# 🗑️ YOLOv8 Waste Detection Model — Google Colab Training

**This notebook trains a custom YOLOv8 model to detect everything that looks like waste:**
- 🛍️ Plastic bags & garbage bags
- 🗑️ Loose garbage & litter
- 📦 Cardboard boxes & cartons
- 🍾 Bottles & cans
- 🧺 Bulk discarded waste

## ⚡ Requirements
- **Runtime**: `Runtime → Change runtime type → T4 GPU` (free)
- **Roboflow API Key**: Free at [https://app.roboflow.com](https://app.roboflow.com) → Settings → API

## 📋 Steps
1. Run all cells top to bottom
2. After training completes, `waste_model.pt` downloads automatically
3. Copy it to `backend/python_detector/waste_model.pt`
4. Update `.env`: `LOCAL_MODEL_PATH=waste_model.pt`


In [ ]:
# ============================================================
# CELL 1: Verify GPU and install dependencies
# ============================================================
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

!pip install ultralytics roboflow --quiet
print('\n✅ Dependencies installed!')

In [ ]:
# ============================================================
# CELL 2: Configure your Roboflow API key
# Get your FREE key at: https://app.roboflow.com/settings/api
# ============================================================
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # <-- PASTE YOUR KEY HERE

if ROBOFLOW_API_KEY == "YOUR_API_KEY_HERE":
    raise ValueError("⚠️  Please replace YOUR_API_KEY_HERE with your actual Roboflow API key!")

print('✅ API key configured.')

In [ ]:
# ============================================================
# CELL 3: Download waste detection dataset from Roboflow Universe
# Using: Garbage Detection dataset (5,000+ annotated images, free)
# Dataset: https://universe.roboflow.com/material-identification/garbage-classification-3
# ============================================================
from roboflow import Roboflow
import os

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Primary dataset: Garbage Classification (bottles, cans, cardboard, plastic)
project = rf.workspace("material-identification").project("garbage-classification-3")
dataset = project.version(12).download("yolov8", location="/content/dataset")

print(f'\n✅ Dataset downloaded to: {dataset.location}')
print(f'Classes: {dataset.classes}')

In [ ]:
# ============================================================
# CELL 4: Remap dataset classes to our 5-class waste schema
# Classes: 0=person, 1=garbage, 2=trash_bag, 3=plastic_bag, 4=waste
# ============================================================
import yaml

# Read the downloaded data.yaml
yaml_path = '/content/dataset/data.yaml'
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

print('Original classes:', data.get('names', []))

# Override with our canonical 5-class schema
data['names'] = {
    0: 'person',
    1: 'garbage',
    2: 'trash_bag',
    3: 'plastic_bag',
    4: 'waste'
}
data['nc'] = 5

# Save updated yaml
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print('\n✅ data.yaml updated to 5-class waste schema:')
print('  0: person')
print('  1: garbage  (loose litter, scattered trash)')
print('  2: trash_bag  (tied black/coloured garbage bags)')
print('  3: plastic_bag  (carrier/shopping bags on ground)')
print('  4: waste  (boxes, bulk discarded items)')

In [ ]:
# ============================================================
# CELL 5: Train YOLOv8s on the waste dataset
# YOLOv8s = small model, best for CCTV surveillance speed/accuracy
# Training time: ~30-45 minutes on T4 GPU
# ============================================================
from ultralytics import YOLO

# Load base pretrained YOLOv8 small model
model = YOLO('yolov8s.pt')

# Fine-tune on waste dataset with CCTV-optimised augmentations
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='waste_dumping_detector',
    device=0,           # GPU
    save=True,
    plots=True,
    patience=15,        # Early stopping
    # Augmentations tuned for outdoor CCTV footage:
    degrees=10.0,       # Slight rotation (camera tilt)
    flipud=0.0,         # No upside-down (gravity exists)
    fliplr=0.5,         # Horizontal flip
    mosaic=1.0,         # Mosaic augmentation
    mixup=0.1,          # Subtle mixup
    hsv_h=0.015,        # Colour variation (different lighting)
    hsv_s=0.7,
    hsv_v=0.4,
)

print('\n✅ Training complete!')
print(f'Best model saved at: {results.save_dir}/weights/best.pt')

In [ ]:
# ============================================================
# CELL 6: Evaluate on validation set
# ============================================================
metrics = model.val()

print('\n📊 Validation Results:')
print(f'  mAP@50:    {metrics.box.map50:.3f}')
print(f'  mAP@50-95: {metrics.box.map:.3f}')
print(f'  Precision: {metrics.box.mp:.3f}')
print(f'  Recall:    {metrics.box.mr:.3f}')

# Show training plots
from IPython.display import Image, display
import os
save_dir = str(results.save_dir)
for plot in ['results.png', 'confusion_matrix.png']:
    path = os.path.join(save_dir, plot)
    if os.path.exists(path):
        display(Image(path))

In [ ]:
# ============================================================
# CELL 7: Test on a sample image (optional visual check)
# ============================================================
import urllib.request
from IPython.display import Image, display

# Download a sample waste image for testing
test_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/7/72/Littering_on_road.jpg/640px-Littering_on_road.jpg'
urllib.request.urlretrieve(test_url, '/content/test_waste.jpg')

# Run inference
best_model = YOLO(str(results.save_dir) + '/weights/best.pt')
test_results = best_model.predict('/content/test_waste.jpg', conf=0.25, save=True)

# Show result
result_img = test_results[0].save_dir + '/' + os.path.basename('/content/test_waste.jpg')
if os.path.exists(result_img):
    display(Image(result_img))
else:
    print('Detections:', [(best_model.names[int(b.cls)], f'{float(b.conf):.2f}') for b in test_results[0].boxes])

In [ ]:
# ============================================================
# CELL 8: Export best.pt and download to your PC
# ============================================================
import shutil
from google.colab import files

best_pt = str(results.save_dir) + '/weights/best.pt'
shutil.copy(best_pt, '/content/waste_model.pt')

size_mb = os.path.getsize('/content/waste_model.pt') / 1024 / 1024
print(f'✅ waste_model.pt ready ({size_mb:.1f} MB)')
print('⬇️  Downloading to your PC now...')

files.download('/content/waste_model.pt')

print('''
=============================================================
  DEPLOYMENT STEPS (run on your PC after downloading)
=============================================================

1. Copy the downloaded file:
   Move waste_model.pt to:
   c:\\Users\\ASUS\\OneDrive\\Desktop\\ibm\\backend\\python_detector\\

2. Edit .env in python_detector/:
   LOCAL_MODEL_PATH=waste_model.pt
   OPEN_VOCABULARY=0
   WASTE_CLASSES=garbage,trash_bag,plastic_bag,waste
   ROBOFLOW_CONFIDENCE=0.35

3. Kill the old detector and restart:
   python detector.py

4. Run CCTV detection:
   python cctv_detector.py --source 0 --display
=============================================================
''')